# PokeAPI

Mariana Oliveira

Carlos Daniel

---

Descrição da atividade: PokeAPI

Utilize as URLs abaixo para praticar acesso a APIs e manipulação de dicionários:

1. https://pokeapi.co/api/v2/pokemon/ditto
   Exiba as habilidades (abilities) do Pokémon.
2. https://pokeapi.co/api/v2/ability/battle-armor
   Mostre o campo short_effect em inglês.
   Liste todos os Pokémons que possuem essa habilidade.

### Bibliotecas

In [102]:
import requests

### Atividades

#### 1. Exiba as habilidades (abilities) do Pokémon.

In [103]:
response = requests.get("https://pokeapi.co/api/v2/pokemon/ditto")
dados_ditto = response.json()

# abilities é uma lista de dicionários, cada um com uma sub-chave "ability"
habilidades = [a["ability"]["name"] for a in dados_ditto["abilities"]]

print("Habilidades do Ditto:")
for h in habilidades:
    print("-", h)

Habilidades do Ditto:
- limber
- imposter


#### 2. Mostre o campo short_effect em inglês.

In [104]:
response = requests.get("https://pokeapi.co/api/v2/ability/battle-armor")
dados_habilidade = response.json()

# effect_entries tem a descrição em vários idiomas, filtramos o inglês
for entry in dados_habilidade["effect_entries"]:
    if entry["language"]["name"] == "en":
        print(entry["short_effect"])

Protects against critical hits.


#### 3. Liste todos os Pokémons que possuem essa habilidade.

In [105]:
pokemons = dados_habilidade["pokemon"]

print(f"Total: {len(pokemons)} pokémons\n")
for p in pokemons:
    print(p["pokemon"]["name"])

Total: 11 pokémons

cubone
marowak
kabuto
kabutops
anorith
armaldo
skorupi
drapion
type-null
perrserker
falinks


# Dados Abertos da Câmara

Mariana Oliveira

Carlos Daniel

---

Documentação: https://dadosabertos.camara.leg.br/swagger/api.html

Dica: mude a saída dos exemplos para JSON

1. Liste os deputados de um determinado partido em um estado específico. Exemplo: deputados do PT no MA.

2. Calcule o total gasto com a cota parlamentar de um deputado em um ano específico. Exemplo: valor líquido dos gastos da deputada Tabata do Amaral em 2026.

3. Liste os fornecedores que mais receberam recursos da cota parlamentar, em ordem decrescente de valor. Utilize um dicionário para acumular os valores por fornecedor. Use sorted(dicionario, key=funcao, reverse=True) para ordenar. Exemplo: maiores fornecedores da deputada Tabata do Amaral em 2026.

4. Analise os discursos de um deputado em um período determinado. Exemplo: deputada Tabata do Amaral de 01/01/2023 até hoje. Use um dicionário para contar a frequência das palavras. Ordene em ordem decrescente de frequência. Remova preposições, conjunções e outras stop words. Sugestão: utilize a biblioteca NLTK.

### Bibliotecas, configurações e utils

#### Bibliotecas

In [106]:
%pip install nltk -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [107]:
import requests
import json
import pandas as pd

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [108]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Mari\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Mari\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Mari\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

#### Utils

In [109]:
BASE_URL = "https://dadosabertos.camara.leg.br/api/v2"

A API pagina os resultados, então essa função vai buscando página por página até
a resposta vir vazia (`dados` sem nada) e junta tudo numa lista só. Quando o endpoint
é de um deputado só (tipo `/deputados/{id}`), a resposta já vem como dicionário, então
a função devolve direto.

In [110]:
def buscar_paginado(endpoint, params=None, itens=100):
  if params is None:
    params = {}

  resultado = []
  pagina = 1

  while True:
    params["pagina"] = pagina
    params["itens"] = itens

    r = requests.get(BASE_URL + endpoint, params=params)
    dados = r.json()["dados"]

    if len(dados) == 0:
      break

    resultado += dados
    pagina += 1

  return resultado

### Atividades

#### 1. Liste os deputados de um determinado partido em um estado específico. Exemplo: deputados do PT no MA.

In [111]:
def deputados_partido_uf(partido, uf):
  params = {"siglaPartido": partido, "siglaUf": uf}
  dados = buscar_paginado("/deputados", params)
  return pd.DataFrame(dados)


df_pt_ma = deputados_partido_uf("PT", "SP")
print("Deputados do PT no SP:")
print(df_pt_ma[["id", "nome", "siglaPartido", "siglaUf"]].to_string(index=False))

Deputados do PT no SP:
    id              nome siglaPartido siglaUf
204501   Alencar Santana           PT      SP
221148        Alfredinho           PT      SP
 73433 Arlindo Chinaglia           PT      SP
141398  Carlos Zarattini           PT      SP
141456      Jilmar Tatto           PT      SP
220640   Juliana Cardoso           PT      SP
162067     Kiko Celeguim           PT      SP
178986       Nilto Tatto           PT      SP
141488    Paulo Teixeira           PT      SP
 73604        Rui Falcão           PT      SP


#### 2. Calcule o total gasto com a cota parlamentar de um deputado em um ano específico. Exemplo: valor líquido dos gastos da deputada Tabata do Amaral em 2026.

Id da deputada Arthur Lira na API da Câmara: **160541**.

In [112]:
id_arthur = 160541

despesas_2026 = buscar_paginado(f"/deputados/{id_arthur}/despesas", {
    "ano": 2026,
    "idLegislatura": 57,
})

total = sum(d["valorLiquido"] for d in despesas_2026)

print(f"Total de despesas: {len(despesas_2026)}")
print("Valor total gasto em 2026: R$", round(total, 2))

Total de despesas: 126
Valor total gasto em 2026: R$ 166884.01


#### 3. Liste os fornecedores que mais receberam recursos da cota parlamentar, em ordem decrescente de valor. Exemplo: maiores fornecedores da deputada Tabata do Amaral em 2026.

In [113]:
gastos_fornecedor = {}

for despesa in despesas_2026:
  nome = despesa["nomeFornecedor"]
  valor = despesa["valorLiquido"]
  gastos_fornecedor[nome] = gastos_fornecedor.get(nome, 0) + valor

ranking = sorted(gastos_fornecedor.items(), key=lambda x: x[1], reverse=True)

for nome, valor in ranking[:15]:
  print(nome, "R$", round(valor, 2))

Gol Linhas Aéreas R$ 54876.27
LATAM Airlines Brasil R$ 47832.32
OK LOCADORA DE VEÍCULOS LTDA - EPP R$ 15000.0
BROAD BRASIL LTDA R$ 5982.0
CDM LOCADORA EM GERAL LTDA R$ 4960.0
TECNEGOCIOS SOLUÇÕES EM INFORMÁTICA LTDA R$ 4000.0
OK LOCADORA DE VEICULOS LTDA R$ 3000.0
POSTO BOULANGERIE LTDA R$ 2304.57
MAXI LUB LTDA R$ 2166.54
TECNEGOCIOS SOLUCOES EM INFORMATICA LTDA R$ 2000.0
TULEMON COMERCIO LTDA R$ 1900.01
Azul Linhas Aéreas R$ 1761.76
POSTO DE COMBUSTIVEL QL 09 LAGO SUL LTDA R$ 1550.0
MAJH COM. DE COMBUST. E DERIV DE PETRO.LTDA EPP R$ 1494.26
AUTO POSTO FAROL LTDA R$ 1490.5


#### 4. Analise os discursos de um deputado em um período determinado. Exemplo: deputada Tabata do Amaral de 01/01/2023 até hoje.

In [114]:
# nome civil, nome parlamentar, partido e UF pra tirar da contagem depois (senão o próprio
# nome da deputada vira "palavra mais frequente", o que não ajuda em nada na análise)
detalhes_arthur = requests.get(BASE_URL + f"/deputados/{id_arthur}").json()["dados"]
status = detalhes_arthur["ultimoStatus"]

termos_proprios = (
    detalhes_arthur["nomeCivil"] + " " +
    status["nome"] + " " +
    status["nomeEleitoral"] + " " +
    status["siglaPartido"] + " " +
    status["siglaUf"]
).lower().split()

termos_proprios = set(termos_proprios)

In [115]:
discursos = buscar_paginado(f"/deputados/{id_arthur}/discursos", {
    "dataInicio": "2023-01-01",
    "dataFim": "2026-12-31",
    "idLegislatura": 57,
})

print(f"Total de discursos: {len(discursos)}")

Total de discursos: 41


In [116]:
stop_words_extra = {
    "sr", "sra", "presidente", "obrigada", "obrigado", "bloco", "ordem",
    "revisão", "aparte", "nobre", "excelência", "deputado", "deputada", "federal",
}

stop_words = set(stopwords.words("portuguese")) | termos_proprios | stop_words_extra

frequencia = {}

for discurso in discursos:
  texto = discurso["transcricao"].lower()
  palavras = word_tokenize(texto, language="portuguese")

  for palavra in palavras:
    if palavra.isalpha() and palavra not in stop_words:
      frequencia[palavra] = frequencia.get(palavra, 0) + 1

ranking_palavras = sorted(frequencia.items(), key=lambda x: x[1], reverse=True)

top_20 = dict(ranking_palavras[:20])
print(json.dumps(top_20, indent=2, ensure_ascii=False))

{
  "casa": 122,
  "deputados": 121,
  "todos": 121,
  "palmas": 92,
  "plenário": 87,
  "brasil": 84,
  "sobre": 76,
  "câmara": 71,
  "desta": 64,
  "aqui": 62,
  "cada": 58,
  "lei": 55,
  "país": 54,
  "reforma": 51,
  "projeto": 50,
  "parlamentares": 50,
  "líderes": 49,
  "hoje": 49,
  "nacional": 49,
  "vamos": 48
}
